# Task A.2.1: Llama 3.1 8B Zero-Shot BHC Baseline

**Task ID:** A.2.1  
**Date created:** 2026-04-29  
**Model:** `meta-llama/Llama-3.1-8B-Instruct` (4-bit quantized)  
**Purpose:** Generate zero-shot Brief Hospital Course summaries for the 100-patient VeriFact-BHC cohort, completing the L-ZS corner of the Phase A 2×2 model-method matrix.

This notebook is a **direct port of `02_baseline.ipynb`**: only the generation model and the output parquet filenames change. Truncation budget (31,000 tokens), generation kwargs, prompt content, and the per-patient output schema are preserved exactly so that L-ZS results are directly comparable to the M-ZS baseline.

# Baseline Pipeline: Zero-Shot BHC Generation
## Step 1: Setup

This notebook implements the baseline approach for Brief Hospital Course generation: concatenate all clinical notes for a patient and prompt an LLM with a simple instruction. No retrieval, no structured template, no examples.

**Environment:** Google Colab Pro (A100 GPU, High-RAM)  
**Data:** VeriFact-BHC processed parquet files (from `01_data_exploration.ipynb`)

In [3]:
# Mount Google Drive to access our processed data files
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Verify we can see our data files
import os
DATA_DIR = "/content/drive/MyDrive/BMI702 Project/Data"

# List files to confirm the path is correct
print("Files in data directory:")
for f in os.listdir(DATA_DIR):
    print(f"  {f}")

Files in data directory:
  propositions_with_gt.parquet
  human_bhcs.parquet
  notes.parquet
  baseline_results.parquet
  baseline_eval_results.parquet
  rag3_matched_budget_results.parquet
  rag3_matched_budget_eval_results.parquet
  rag3_proposition_eval.parquet
  rag_results.parquet
  rag_eval_results.parquet
  rag2_multi_query_results.parquet
  rag2_multi_query_eval_results.parquet
  proposition_eval_results.parquet
  llm_judge_single_summary_scores.parquet
  llm_judge_pairwise_scores.parquet
  llm_judge_summary_table.csv
  llm_judge_pairwise_winner_counts.csv
  llm_judge_plus_auto_metrics_merged.parquet
  llm_judge_strict_with_rag3_pairwise_winner_counts_first5.csv
  llm_judge_strict_with_rag3_summary_table_first5.csv
  llm_judge_strict_with_rag3_merged_first5.parquet
  llm_judge_strict_with_rag3_single_scores_first5.parquet
  llm_judge_strict_with_rag3_pairwise_first5.parquet
  llm_judge_qwen3_32b_jsonfix_with_rag3_single_scores_first5.parquet
  llm_judge_qwen3_32b_jsonfix_with_r

### 1.1 Load Processed Data

In [5]:
# Load the processed data from our exploration notebook
import pandas as pd
import numpy as np

notes = pd.read_parquet(f"{DATA_DIR}/notes.parquet")
human_bhcs = pd.read_parquet(f"{DATA_DIR}/human_bhcs.parquet")

print(f"Notes: {notes.shape[0]} rows")
print(f"BHC targets: {human_bhcs.shape[0]} patients")

Notes: 4787 rows
BHC targets: 100 patients


### 1.2 Install Dependencies and Load Model

We use **Llama 3.1 8B Instruct** as the second generation model in Phase A. The Mistral 7B baseline (`02_baseline.ipynb`) is the comparison condition; this notebook completes the L-ZS corner of the Phase A 2×2 matrix.

**Llama 3.1 8B Instruct:**
- Open-weights but **gated** — requires HuggingFace license acceptance before download
- 8B parameters — fits on T4/A100 GPU at 4-bit quantization (~5GB VRAM)
- 128K native context window — substantially larger than Mistral's 32K, but **we hold the input budget at 31,000 tokens to match the Mistral baseline exactly**. Whether to take advantage of Llama's full context is a separate experiment, not part of this port.

The 32K-equivalent context limit is itself a motivation for the RAG approach, which selects only the most relevant content rather than trying to fit everything into one prompt.

In [6]:
# Install required packages for model loading and quantization
!pip install -q transformers accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.9 MB/s eta 0:00:00


In [7]:
# Log in to Hugging Face — required for gated Llama 3.1 access
from huggingface_hub import login
login()

In [9]:
# Load Llama 3.1 8B Instruct with the same 4-bit quantization config as the Mistral baseline.
# NF4 + fp16 compute + double-quant transfers cleanly to Llama 3.1 8B (same memory profile, ~5GB).
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "meta-llama/Llama-3.1-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print(f"Model loaded: {model_name}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded: meta-llama/Llama-3.1-8B-Instruct
GPU memory used: 5.7 GB


## Step 2: Build the Baseline Pipeline

The baseline approach is deliberately simple: for each patient, concatenate all clinical notes chronologically and ask the model to summarize the hospital course. No retrieval, no structure, no examples. This isolates what a general-purpose LLM can produce from raw input alone.

In [10]:
# First, let's prepare the input for each patient
# Concatenate all notes chronologically, separated by headers showing note type and date

def prepare_patient_input(patient_id, notes_df):
    """
    Concatenate all notes for a patient in chronological order.
    Each note gets a header with its category and date for context.
    """
    # Get all notes for this patient, sorted by date
    patient_notes = (
        notes_df[notes_df['SUBJECT_ID'] == patient_id]
        .sort_values(['CHARTDATE', 'CHARTTIME'])
    )

    # Build the concatenated text with note headers
    sections = []
    for _, row in patient_notes.iterrows():
        header = f"[{row['CATEGORY']} — {row['CHARTDATE']}]"
        sections.append(f"{header}\n{row['TEXT']}")

    return "\n\n".join(sections)

# Test on one patient to see what the input looks like
test_patient = human_bhcs['subject_id'].iloc[0]
test_input = prepare_patient_input(test_patient, notes)

# Count tokens using the actual tokenizer
test_tokens = tokenizer.encode(test_input)
print(f"Patient {test_patient}: {len(test_tokens)} tokens across "
      f"{len(notes[notes['SUBJECT_ID'] == test_patient])} notes")
print(f"\nFirst 500 characters of concatenated input:")
print(test_input[:500])

Patient 1084: 13361 tokens across 11 notes

First 500 characters of concatenated input:
[Physician — 2198-08-02]
Chief Complaint:  Primary Care Physician: [**Name10 (NameIs) **],[**Known firstname **] [**Name Initial (NameIs) **] [**Telephone/Fax (1) 9915**]
   .
   Chief Complaint: AMS
   HPI:
   Pt is a 61 y.o male with h.o prostate ca, PE, nephrolithiasis, pelvic
   osteo, HL, chronic back and LLE pain, [**Doctor Last Name **], UTIs, urethral
   dilatation, multiple leg and pelvic debridements who was BIBA to the ED
   reportedly after construction workers found him "acting funn


In [11]:
# Define the zero-shot prompt
# No examples, no structure, no guidance on format
# Preserved byte-for-byte from 02_baseline.ipynb so the L-ZS condition differs from M-ZS only in the model.

BASELINE_PROMPT = """You are a physician. Based on the following clinical notes from a patient's hospital stay, write a Brief Hospital Course summarizing the key events, findings, treatments, and outcomes.

Clinical Notes:
{notes}

Brief Hospital Course:"""

In [12]:
def generate_bhc(patient_input, model, tokenizer, max_context=31000, max_new_tokens=1024):
    """
    Generate a Brief Hospital Course from concatenated clinical notes.

    Args:
        patient_input: Concatenated clinical notes text
        model: The loaded LLM
        tokenizer: The tokenizer
        max_context: Maximum tokens for input (leaving room for prompt + generation)
        max_new_tokens: Maximum tokens to generate

    Returns:
        generated_text: The model's BHC output
        was_truncated: Whether the input had to be truncated
    """
    # Build the full prompt
    full_prompt = BASELINE_PROMPT.format(notes=patient_input)

    # Tokenize and check length
    tokens = tokenizer.encode(full_prompt)
    was_truncated = False

    # If too long, truncate the notes (keep the prompt structure intact)
    # Budget held at 31000 tokens to match the Mistral baseline (Task A.2.1).
    # Truncation is applied to the unformatted prompt; the chat-template tokens are added afterward.
    if len(tokens) > max_context:
        was_truncated = True
        # Truncate from the end — keep earliest notes, lose most recent
        # (A simple strategy; we could also truncate from the middle or prioritize note types)
        tokens = tokens[:max_context]
        full_prompt = tokenizer.decode(tokens, skip_special_tokens=True)

    # Format as a chat message for the instruct model.
    # Single user message with no system role — the original Mistral notebook embedded
    # "You are a physician..." inside the user content because Mistral v0.3 has no system role.
    # We preserve that structure exactly here (Task A.2.1 D1: Option A) so the L-ZS vs M-ZS
    # comparison varies only the model. apply_chat_template handles the model-specific format.
    messages = [{"role": "user", "content": full_prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize the formatted prompt
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,        # Low temperature for consistent, factual output
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1   # Discourage repetitive text
        )

    # Decode only the generated tokens (not the input prompt)
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return generated_text, was_truncated

In [13]:
# Test on our example patient before running the full cohort
print(f"Generating BHC for patient {test_patient} with {model_name}...")
print(f"Input: {len(tokenizer.encode(test_input))} tokens")

generated_bhc, was_truncated = generate_bhc(test_input, model, tokenizer)

print(f"Truncated: {was_truncated}")
print(f"Generated BHC ({len(tokenizer.encode(generated_bhc))} tokens):")
print("=" * 80)
print(generated_bhc)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generating BHC for patient 1084 with meta-llama/Llama-3.1-8B-Instruct...
Input: 13361 tokens
Truncated: False
Generated BHC (542 tokens):
**Patient Information**

* Name: [**Name10 (NameIs)**]
* Age: 61 years old
* Sex: Male
* Chief Complaint: Altered Mental Status (AMS)

**Admission**

* Date: August 2, 2198
* Time: Unknown
* Mode of Arrival: Emergency Department (ED)
* Reason for Admission: AMS, suspected opioid withdrawal, and potential urinary tract infection (UTI)

**History of Present Illness**

* The patient was found by construction workers to be altered and naked at his home.
* He was given naloxone (narcan) by emergency medical services (EMS), which led to increased agitation and confusion.
* In the ED, he was intubated due to severe agitation and received multiple medications, including benzodiazepines and opioids.

**Initial Findings**

* Head CT scan: Negative for acute process
* Urinalysis (UA): Positive for nitrates, leukocytes, and bacteria
* Toxicology screen: Positive

### 2.1 Qualitative Check: Compare Generated vs. Human BHC

Before running the full cohort, let's compare the generated output against the gold standard for our test patient. This helps us calibrate expectations.

In [14]:
# Side-by-side comparison: generated vs. human-written BHC
human_bhc_text = human_bhcs[human_bhcs['subject_id'] == test_patient]['brief_hospital_course'].iloc[0]

print("HUMAN-WRITTEN BHC")
print("=" * 80)
print(human_bhc_text[:2000])
print("..." if len(human_bhc_text) > 2000 else "")
print(f"\n({len(tokenizer.encode(human_bhc_text))} tokens)")

print("\n\nGENERATED BHC (Llama 3.1 8B, zero-shot)")
print("=" * 80)
print(generated_bhc)
print(f"\n({len(tokenizer.encode(generated_bhc))} tokens)")

HUMAN-WRITTEN BHC
Pt is a 61 y.o male with h.o prostate ca with fistulous and
infectious complications, pelvis osteo, PE, nephrolithiasis, HL,
chronic back and LLE pain, [**Doctor Last Name 933**], UTIs, multiple leg
debridements with presents with AMS.

# AMS- secondary to medication error and exacerbated by opiate
withdrawal.  The patient was found altered at home, given narcan
and woke up to become very combative and difficult to control.
He was treated with several medications in the ED, but needed to
be intubated to have a CT scan and LP.  His CT and LP were both
negative.  He was monitored overnight and treated with zosyn for
a supicious UA.  In the AM of his second day of hospitalization,
he woke up and self-extubated himself despite being on a
propofol gtt.  His mental status began to clear over the course
of the day, and he remember that instead of taking his 2.5 tabs
of methadone, he took 2.5 tabs of ambien.  That was likely the
cause of his altered mental status.  He was app

## Step 3: Run Baseline on All 100 Patients

Generate zero-shot BHCs for all 100 patients using Llama 3.1 8B. We track which patients required truncation and the generation time per patient.

In [15]:
# Smoke-test switch: set LIMIT_PATIENTS = 5 (or any small N) to run on the first
# N patients before committing to the full cohort. Leave as None for the real run.
# This convention is shared with 03b_rag_llama.ipynb and downstream A.2 notebooks (per Task A.2.1 D2).
LIMIT_PATIENTS = None  # set to 5 for smoke test

In [16]:
import time

# Store results for all patients
results = []

# Honor the smoke-test switch from the cell above
patients_to_run = human_bhcs.head(LIMIT_PATIENTS) if LIMIT_PATIENTS else human_bhcs

print(f"Running baseline on {len(patients_to_run)} patients...")
print("-" * 60)

start_total = time.time()

for i, row in patients_to_run.iterrows():
    patient_id = row['subject_id']

    # Prepare concatenated input
    patient_input = prepare_patient_input(patient_id, notes)
    input_tokens = len(tokenizer.encode(patient_input))

    # Generate BHC
    start = time.time()
    generated, was_truncated = generate_bhc(patient_input, model, tokenizer)
    elapsed = time.time() - start

    # Store results
    results.append({
        'subject_id': patient_id,
        'input_tokens': input_tokens,
        'was_truncated': was_truncated,
        'generated_bhc': generated,
        'generated_tokens': len(tokenizer.encode(generated)),
        'generation_time_sec': round(elapsed, 1),
        'human_bhc': row['brief_hospital_course']
    })

    # Progress update every 10 patients
    if (len(results)) % 10 == 0:
        print(f"  {len(results)}/{len(patients_to_run)} patients complete | "
              f"Last: {input_tokens} input tokens, {elapsed:.1f}s | "
              f"Truncated: {was_truncated}")

total_time = time.time() - start_total
print("-" * 60)
print(f"Done! Total time: {total_time/60:.1f} minutes")
print(f"Patients truncated: {sum(r['was_truncated'] for r in results)}/{len(patients_to_run)}")

# Convert to dataframe
results_df = pd.DataFrame(results)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running baseline on 100 patients...
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Token indices sequence length is longer than the specified maximum sequence length for this model (237336 > 131072). Running this sequence through the model will result in indexing errors
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  10/100 patients complete | Last: 79260 input tokens, 37.5s | Truncated: True


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

  80/100 patients complete | Last: 19129 input tokens, 47.6s | Truncated: False


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  90/100 patients complete | Last: 18511 input tokens, 47.8s | Truncated: False


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  100/100 patients complete | Last: 22518 input tokens, 61.9s | Truncated: False
------------------------------------------------------------
Done! Total time: 85.0 minutes
Patients truncated: 34/100


In [17]:
# Save results to Drive right away — don't risk losing an hour of work
results_df.to_parquet(f"{DATA_DIR}/L_ZS_BHC_results.parquet", index=False)
print(f"Saved {len(results_df)} results to Drive")

Saved 100 results to Drive


## Step 4: Baseline Results Summary

In [18]:
# Quick overview of the generation run
print("BASELINE RUN SUMMARY")
print("=" * 60)
print(f"Total patients: {len(results_df)}")
print(f"Patients truncated: {results_df['was_truncated'].sum()}/{len(results_df)}")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"\nInput tokens (all patients):")
print(f"  Mean:   {results_df['input_tokens'].mean():.0f}")
print(f"  Median: {results_df['input_tokens'].median():.0f}")
print(f"\nGenerated BHC length (tokens):")
print(f"  Mean:   {results_df['generated_tokens'].mean():.0f}")
print(f"  Median: {results_df['generated_tokens'].median():.0f}")
print(f"\nHuman BHC length (tokens) for comparison:")
human_tokens = results_df['human_bhc'].apply(lambda x: len(tokenizer.encode(x)))
print(f"  Mean:   {human_tokens.mean():.0f}")
print(f"  Median: {human_tokens.median():.0f}")

BASELINE RUN SUMMARY
Total patients: 100
Patients truncated: 34/100
Total time: 85.0 minutes

Input tokens (all patients):
  Mean:   44699
  Median: 20692

Generated BHC length (tokens):
  Mean:   566
  Median: 537

Human BHC length (tokens) for comparison:
  Mean:   641
  Median: 530


## Step 5: Evaluation — Automated Metrics

We evaluate generated BHCs against human-written gold standards using:
- **ROUGE-L**: Measures longest common subsequence overlap (lexical similarity)
- **BERTScore**: Measures semantic similarity using contextual embeddings (captures meaning even when wording differs)

These are the standard metrics used in clinical summarization research (Van Veen et al. 2024, Xu et al. 2024 "Discharge Me!" shared task).

In [19]:
# Install evaluation packages
!pip install -q rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.3 MB/s eta 0:00:00


In [20]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Compute ROUGE scores for all patients
rouge_results = []
for _, row in results_df.iterrows():
    scores = scorer.score(row['human_bhc'], row['generated_bhc'])
    rouge_results.append({
        'subject_id': row['subject_id'],
        'rouge1_f': scores['rouge1'].fmeasure,
        'rouge2_f': scores['rouge2'].fmeasure,
        'rougeL_f': scores['rougeL'].fmeasure,
        'was_truncated': row['was_truncated']
    })

rouge_df = pd.DataFrame(rouge_results)

print(f"ROUGE Scores (baseline, all {len(rouge_df)} patients):")
print(f"  ROUGE-1 F1: {rouge_df['rouge1_f'].mean():.3f} (±{rouge_df['rouge1_f'].std():.3f})")
print(f"  ROUGE-2 F1: {rouge_df['rouge2_f'].mean():.3f} (±{rouge_df['rouge2_f'].std():.3f})")
print(f"  ROUGE-L F1: {rouge_df['rougeL_f'].mean():.3f} (±{rouge_df['rougeL_f'].std():.3f})")

ROUGE Scores (baseline, all 100 patients):
  ROUGE-1 F1: 0.340 (±0.067)
  ROUGE-2 F1: 0.079 (±0.035)
  ROUGE-L F1: 0.159 (±0.036)


In [21]:
# Compute BERTScore — this takes a couple minutes
# Using the default roberta-large model for English
print("Computing BERTScore (this may take a few minutes)...")

P, R, F1 = bert_score_fn(
    results_df['generated_bhc'].tolist(),
    results_df['human_bhc'].tolist(),
    lang='en',
    verbose=True
)

results_df['bertscore_p'] = P.numpy()
results_df['bertscore_r'] = R.numpy()
results_df['bertscore_f1'] = F1.numpy()

print(f"\nBERTScore (baseline, all {len(results_df)} patients):")
print(f"  Precision: {results_df['bertscore_p'].mean():.3f} (±{results_df['bertscore_p'].std():.3f})")
print(f"  Recall:    {results_df['bertscore_r'].mean():.3f} (±{results_df['bertscore_r'].std():.3f})")
print(f"  F1:        {results_df['bertscore_f1'].mean():.3f} (±{results_df['bertscore_f1'].std():.3f})")

Computing BERTScore (this may take a few minutes)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 3.60 seconds, 27.78 sentences/sec

BERTScore (baseline, all 100 patients):
  Precision: 0.815 (±0.017)
  Recall:    0.810 (±0.013)
  F1:        0.813 (±0.013)


### 5.1 Impact of Truncation on Quality

A substantial fraction of patients (42/100 in the Mistral baseline) required truncation to fit the 31K-token budget. Do truncated patients produce worse summaries? If so, this directly motivates the RAG approach.

In [22]:
# Merge ROUGE scores with results for analysis
eval_df = results_df.merge(rouge_df, on='subject_id')

# Compare truncated vs non-truncated patients
trunc = eval_df[eval_df['was_truncated_x'] == True]
no_trunc = eval_df[eval_df['was_truncated_x'] == False]

print("TRUNCATED vs NON-TRUNCATED PATIENTS")
print("=" * 60)
print(f"{'Metric':<20} {f'Truncated (n={len(trunc)})':<22} {f'Not truncated (n={len(no_trunc)})'}")
print("-" * 60)
print(f"{'ROUGE-1 F1':<20} {trunc['rouge1_f'].mean():.3f} (±{trunc['rouge1_f'].std():.3f})     "
      f"{no_trunc['rouge1_f'].mean():.3f} (±{no_trunc['rouge1_f'].std():.3f})")
print(f"{'ROUGE-L F1':<20} {trunc['rougeL_f'].mean():.3f} (±{trunc['rougeL_f'].std():.3f})     "
      f"{no_trunc['rougeL_f'].mean():.3f} (±{no_trunc['rougeL_f'].std():.3f})")
print(f"{'BERTScore F1':<20} {trunc['bertscore_f1'].mean():.3f} (±{trunc['bertscore_f1'].std():.3f})     "
      f"{no_trunc['bertscore_f1'].mean():.3f} (±{no_trunc['bertscore_f1'].std():.3f})")
print(f"{'Input tokens':<20} {trunc['input_tokens'].mean():.0f}               "
      f"{no_trunc['input_tokens'].mean():.0f}")

TRUNCATED vs NON-TRUNCATED PATIENTS
Metric               Truncated (n=34)       Not truncated (n=66)
------------------------------------------------------------
ROUGE-1 F1           0.322 (±0.066)     0.350 (±0.066)
ROUGE-L F1           0.141 (±0.026)     0.168 (±0.036)
BERTScore F1         0.810 (±0.011)     0.814 (±0.013)
Input tokens         100268               16073


In [23]:
# Save all evaluation results to Drive
eval_df.to_parquet(f"{DATA_DIR}/L_ZS_BHC_eval_results.parquet", index=False)
print(f"Saved evaluation results to Drive")

# summary
print(f"""
Model: Llama 3.1 8B Instruct (4-bit quantized)
Patients: {len(results_df)} (VeriFact-BHC cohort)
Truncated: {results_df['was_truncated'].sum()}/{len(results_df)} (31K-token budget)
Runtime: {total_time/60:.1f} minutes on A100

ROUGE-1 F1: {rouge_df['rouge1_f'].mean():.3f} (±{rouge_df['rouge1_f'].std():.3f})
ROUGE-2 F1: {rouge_df['rouge2_f'].mean():.3f} (±{rouge_df['rouge2_f'].std():.3f})
ROUGE-L F1: {rouge_df['rougeL_f'].mean():.3f} (±{rouge_df['rougeL_f'].std():.3f})
BERTScore F1: {results_df['bertscore_f1'].mean():.3f} (±{results_df['bertscore_f1'].std():.3f})

Generated BHC length: mean {results_df['generated_tokens'].mean():.0f} tokens
Human BHC length: mean {human_tokens.mean():.0f} tokens
""")

Saved evaluation results to Drive

Model: Llama 3.1 8B Instruct (4-bit quantized)
Patients: 100 (VeriFact-BHC cohort)
Truncated: 34/100 (31K-token budget)
Runtime: 85.0 minutes on A100

ROUGE-1 F1: 0.340 (±0.067)
ROUGE-2 F1: 0.079 (±0.035)
ROUGE-L F1: 0.159 (±0.036)
BERTScore F1: 0.813 (±0.013)

Generated BHC length: mean 566 tokens
Human BHC length: mean 641 tokens

